# Хар. набор для F4

In [ ]:
import numpy as np
from itertools import combinations, product
from math import gcd
from functools import reduce
from fractions import Fraction
import sympy as sp

# ─────────────────────────────────────────────
# Вспомогательные функции
# ─────────────────────────────────────────────

def vector_gcd(v):
    """НОД абсолютных значений ненулевых целых координат вектора."""
    nonzero = [abs(int(x)) for x in v if x != 0]
    if not nonzero:
        return 1
    return reduce(gcd, nonzero)


def frac_vec(v):
    """Перевод numpy-вектора в список Fraction."""
    return [Fraction(x).limit_denominator(1000) for x in v]


def normalize_rational(v):
    """
    Нормализация вектора из Fraction: только делим на НОД координат.
    Знак НЕ фиксируется — v и -v остаются разными элементами набора.
    """
    # Приводим к общему знаменателю
    denom = reduce(lambda a, b: a * b // gcd(a, b), [x.denominator for x in v])
    ints = [int(x * denom) for x in v]

    # Делим на НОД
    g = vector_gcd(ints)
    if g == 0:
        return tuple(Fraction(0) for _ in v)
    ints = [x // g for x in ints]

    return tuple(Fraction(x) for x in ints)


# ─────────────────────────────────────────────
# Генерация корней F4
# ─────────────────────────────────────────────

def generate_F4_roots():
    roots = set()

    # ±e_i
    for i in range(4):
        for sign in [1, -1]:
            v = [0, 0, 0, 0]
            v[i] = sign
            roots.add(tuple(Fraction(x) for x in v))

    # ±e_i ± e_j
    for i in range(4):
        for j in range(i + 1, 4):
            for s1 in [1, -1]:
                for s2 in [1, -1]:
                    v = [0, 0, 0, 0]
                    v[i] = s1
                    v[j] = s2
                    roots.add(tuple(Fraction(x) for x in v))

    # 1/2 (±e1 ± e2 ± e3 ± e4)
    for signs in product([1, -1], repeat=4):
        v = tuple(Fraction(s, 2) for s in signs)
        roots.add(v)

    # Убираем нулевой вектор (на всякий случай)
    roots.discard(tuple(Fraction(0) for _ in range(4)))

    return [list(r) for r in roots]


# ─────────────────────────────────────────────
# Группа Вейля
# ─────────────────────────────────────────────

def reflection_matrix_rational(alpha):
    """Матрица отражения s_alpha как список списков Fraction."""
    n = len(alpha)
    norm_sq = sum(x * x for x in alpha)
    mat = []
    for i in range(n):
        row = []
        for j in range(n):
            delta = Fraction(1) if i == j else Fraction(0)
            row.append(delta - Fraction(2) * alpha[i] * alpha[j] / norm_sq)
        mat.append(row)
    return mat


def mat_mul_rational(A, B):
    """Перемножение квадратных матриц из Fraction."""
    n = len(A)
    C = [[Fraction(0)] * n for _ in range(n)]
    for i in range(n):
        for k in range(n):
            if A[i][k] == 0:
                continue
            for j in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C


def mat_eq(A, B):
    for i in range(len(A)):
        for j in range(len(A[0])):
            if A[i][j] != B[i][j]:
                return False
    return True


def apply_matrix(M, v):
    """Применение матрицы M (список списков Fraction) к вектору v (список Fraction)."""
    n = len(v)
    return [sum(M[i][j] * v[j] for j in range(n)) for i in range(n)]


def generate_weyl_group(roots):
    """BFS-генерация группы Вейля через отражения. Возвращает список матриц."""
    n = 4
    identity = [[Fraction(1) if i == j else Fraction(0) for j in range(n)] for i in range(n)]

    reflections = [reflection_matrix_rational(alpha) for alpha in roots]

    group = [identity]
    to_process = [identity]

    print("  [Вейль] Начинаем BFS...")
    while to_process:
        g = to_process.pop()
        for ref in reflections:
            new_g = mat_mul_rational(ref, g)
            if not any(mat_eq(new_g, existing) for existing in group):
                group.append(new_g)
                to_process.append(new_g)
        if len(group) % 100 == 0:
            print(f"  [Вейль] Найдено элементов: {len(group)}, в очереди: {len(to_process)}")

    return group


# ─────────────────────────────────────────────
# Камеры Вейля
# ─────────────────────────────────────────────

def generate_weyl_chambers(dominant_chamber, weyl_group):
    """Применяем каждый элемент группы к образующим доминантной камеры."""
    chambers = []
    for g in weyl_group:
        chamber = [apply_matrix(g, v) for v in dominant_chamber]
        chambers.append(chamber)
    return chambers


# ─────────────────────────────────────────────
# Характеристический набор — ядро через sympy
# ─────────────────────────────────────────────

def null_space_rational(rows):
    """
    Вычисляет ядро матрицы rows (список списков Fraction) средствами sympy
    (точная арифметика над QQ). Возвращает список векторов-столбцов ядра.
    """
    sp_mat = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in row]
                        for row in rows])
    return sp_mat.nullspace()   # список sp.Matrix-столбцов


def characteristic_set(chambers):
    # Собираем все уникальные рёбра (образующие) камер
    edges_set = set()
    for chamber in chambers:
        for v in chamber:
            key = normalize_rational(v)
            edges_set.add(key)

    edges = [list(k) for k in edges_set]
    print(f"Количество уникальных рёбер: {len(edges)}")

    l = 4          # ранг F4
    k = l - 1      # размер комбинации

    characteristic = set()
    total = sum(1 for _ in combinations(range(len(edges)), k))
    print(f"Всего комбинаций из {k} рёбер: {total}")

    for idx, combo_indices in enumerate(combinations(range(len(edges)), k)):
        if idx % 100_000 == 0 and idx > 0:
            print(f"  Обработано {idx} / {total} комбинаций")

        rows = [edges[i] for i in combo_indices]

        # Ранг через sympy (точно)
        sp_mat = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in row]
                            for row in rows])

        if sp_mat.rank() < k:
            continue   # строки линейно зависимы — пропускаем

        # Ядро: ищем вектор v такой, что rows @ v = 0
        null = sp_mat.nullspace()
        if not null:
            continue

        for ns_vec in null:
            # ns_vec — sympy Matrix (4×1)
            coords = [Fraction(int(x.p), int(x.q)) for x in ns_vec]

            # Нормализация по НОД
            key_pos = normalize_rational(coords)
            key_neg = normalize_rational([-c for c in coords])

            characteristic.add(key_pos)
            characteristic.add(key_neg)

    return [list(v) for v in characteristic]


# ─────────────────────────────────────────────
# Доминантная камера F4 (образующие)
# ─────────────────────────────────────────────

dominant_chamber_F4 = [
    [Fraction(1), Fraction(0), Fraction(0), Fraction(0)],           # e1
    [Fraction(1), Fraction(1), Fraction(0), Fraction(0)],           # e1 + e2
    [Fraction(2), Fraction(1), Fraction(1), Fraction(0)],           # 2e1 + e2 + e3
    [Fraction(3,2), Fraction(1,2), Fraction(1,2), Fraction(1,2)],   # 1/2(3e1+e2+e3+e4)
]

# ─────────────────────────────────────────────
# Основной запуск
# ─────────────────────────────────────────────

print("Генерация корней F4...")
roots = generate_F4_roots()
print(f"Количество корней: {len(roots)}")

print("\nГенерация группы Вейля (рациональная арифметика)...")
weyl_group = generate_weyl_group(roots)
print(f"Порядок группы Вейля: {len(weyl_group)}")

print("\nГенерация камер Вейля...")
chambers = generate_weyl_chambers(dominant_chamber_F4, weyl_group)
print(f"Количество камер Вейля: {len(chambers)}")

print("\nПостроение характеристического набора...")
char_set = characteristic_set(chambers)
print(f"\nКоличество векторов в характеристическом наборе: {len(char_set)}")

with open('f4_characteristic_set_rational3.txt', 'w') as f:
    for v in char_set:
        # f.write("  ".join(str(c) for c in v) + "\n")
        f.write("  ".join(f"{float(c):.10f}" for c in v) + "\n")

print("Результат сохранён в 'f4_characteristic_set_rational3.txt'")

#фильтрация + разнесение, проверка для F4

In [ ]:
import sympy as sp
import itertools
import time
from fractions import Fraction
from math import gcd
from functools import reduce


# =============================================================================
# F4 (Бурбаки)
# =============================================================================

SIMPLE_ROOTS = [
    [Fraction(0), Fraction(1), Fraction(-1), Fraction(0)],
    [Fraction(0), Fraction(0), Fraction(1), Fraction(-1)],
    [Fraction(0), Fraction(0), Fraction(0), Fraction(1)],
    [Fraction(1, 2), Fraction(-1, 2), Fraction(-1, 2), Fraction(-1, 2)],
]

FUND_WEIGHTS = [
    [Fraction(1), Fraction(1), Fraction(0), Fraction(0)],
    [Fraction(2), Fraction(1), Fraction(1), Fraction(0)],
    [Fraction(3, 2), Fraction(1, 2), Fraction(1, 2), Fraction(1, 2)],
    [Fraction(1), Fraction(0), Fraction(0), Fraction(0)],
]


# =============================================================================
# Нормализация (как у тебя в эталоне)
# =============================================================================

def vec_gcd(ints):
    nz = [abs(int(x)) for x in ints if x != 0]
    return reduce(gcd, nz) if nz else 1

def normalize(v):
    """v: список Fraction. Делим на НОД, знак сохраняем."""
    fracs = [Fraction(x) for x in v]
    denom = reduce(lambda a, b: a * b // gcd(a, b),
                   [x.denominator for x in fracs])
    ints = [int(x * denom) for x in fracs]
    g = vec_gcd(ints)
    if g == 0:
        return tuple(Fraction(0) for _ in v)
    return tuple(Fraction(x, g) for x in ints)


# =============================================================================
# Корни F4
# =============================================================================

def f4_roots():
    R = set()
    for i in range(4):
        for s in (1, -1):
            v = [Fraction(0)] * 4
            v[i] = Fraction(s)
            R.add(tuple(v))
    for i, j in itertools.combinations(range(4), 2):
        for si, sj in itertools.product((1, -1), repeat=2):
            v = [Fraction(0)] * 4
            v[i], v[j] = Fraction(si), Fraction(sj)
            R.add(tuple(v))
    for signs in itertools.product((1, -1), repeat=4):
        R.add(tuple(Fraction(s, 2) for s in signs))
    return [list(r) for r in R]


# =============================================================================
# Группа Вейля
# =============================================================================

def reflection_mat(alpha):
    n = len(alpha)
    nrm = sum(x * x for x in alpha)
    return tuple(tuple(
        (Fraction(1) if i == j else Fraction(0)) - Fraction(2) * alpha[i] * alpha[j] / nrm
        for j in range(n)) for i in range(n))

def matmul(A, B):
    n = len(A)
    return tuple(tuple(sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n))
                 for i in range(n))

def matvec(M, v):
    return [sum(M[i][j] * v[j] for j in range(len(v))) for i in range(len(M))]

def build_W(roots):
    n = 4
    I = tuple(tuple(Fraction(1) if i == j else Fraction(0) for j in range(n)) for i in range(n))
    refls = [reflection_mat(a) for a in roots]
    seen = {I}
    frontier = [I]
    while frontier:
        new_frontier = []
        for M in frontier:
            for r in refls:
                M2 = matmul(r, M)
                if M2 not in seen:
                    seen.add(M2)
                    new_frontier.append(M2)
        frontier = new_frontier
    return list(seen)


# =============================================================================
# Множество S
# =============================================================================

def build_S(W, fund):
    keys = set()
    for w in fund:
        for M in W:
            keys.add(normalize(matvec(M, w)))
    return [list(k) for k in keys]


# =============================================================================
# Разложение по базису простых корней (точно, через sympy)
# =============================================================================

# Матрица: столбцы — простые корни. Обратная даёт коэффициенты разложения.
_A_sympy = sp.Matrix([[sp.Rational(SIMPLE_ROOTS[j][i].numerator,
                                   SIMPLE_ROOTS[j][i].denominator)
                       for j in range(4)] for i in range(4)])
_A_inv_sympy = _A_sympy.inv()


def coords_in_simple_root_basis(v):
    """Возвращает список Fraction — координаты v в базисе {alpha_i}."""
    v_sp = sp.Matrix([sp.Rational(x.numerator, x.denominator) for x in v])
    c = _A_inv_sympy * v_sp
    return [Fraction(int(x.p), int(x.q)) for x in c]


def is_strictly_inside_simple_cone_or_opposite(v):
    """True, если все коэффициенты разложения v по {alpha_i} строго одного знака."""
    c = coords_in_simple_root_basis(v)
    if all(x > 0 for x in c):
        return True
    if all(x < 0 for x in c):
        return True
    return False


# =============================================================================
# Принадлежность \bar C
# =============================================================================

def in_dom_closure(u):
    """u in \\bar C <=> <alpha_i, u> >= 0 для всех i."""
    for a in SIMPLE_ROOTS:
        if sum(a[i] * u[i] for i in range(4)) < 0:
            return False
    return True


# =============================================================================
# Базовый алгоритм
# =============================================================================

def alg_base(S, l=4):
    out = set()
    n_combos = 0
    for combo in itertools.combinations(range(len(S)), l - 1):
        n_combos += 1
        rows = [S[i] for i in combo]
        M = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in r]
                       for r in rows])
        if M.rank() < l - 1:
            continue
        for ns in M.nullspace():
            coords = [Fraction(int(x.p), int(x.q)) for x in ns]
            out.add(normalize(coords))
            out.add(normalize([-c for c in coords]))
    return out, n_combos


# =============================================================================
# Идея 1 (исправленная) + отсечение по \bar C + разнесение W
# =============================================================================

def alg_idea1(S, W, l=4):
    # Шаг 1: фильтрация S
    S_filt = []
    for v in S:
        if not is_strictly_inside_simple_cone_or_opposite(v):
            S_filt.append(v)

    # Шаг 2: перебор троек, отбор пересечений в \bar C
    partial = set()
    n_combos = 0
    for combo in itertools.combinations(range(len(S_filt)), l - 1):
        n_combos += 1
        rows = [S_filt[i] for i in combo]
        M = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in r]
                       for r in rows])
        if M.rank() < l - 1:
            continue
        for ns in M.nullspace():
            u = [Fraction(int(x.p), int(x.q)) for x in ns]
            mu = [-c for c in u]
            if in_dom_closure(u):
                partial.add(normalize(u))
            if in_dom_closure(mu):
                partial.add(normalize(mu))

    # Шаг 3: разнесение группой W
    full = set()
    for k in partial:
        v = list(k)
        for Mw in W:
            full.add(normalize(matvec(Mw, v)))

    return full, len(S_filt), len(partial), n_combos


# =============================================================================
# Main
# =============================================================================

def main():
    print("[1] корни F4")
    R = f4_roots()
    print(f"    |R| = {len(R)}")

    print("[2] W")
    t0 = time.time()
    W = build_W(R)
    print(f"    |W| = {len(W)}   {time.time()-t0:.2f}с")

    print("[3] S")
    t0 = time.time()
    S = build_S(W, FUND_WEIGHTS)
    print(f"    |S| = {len(S)}   {time.time()-t0:.2f}с")

    # Диагностика: разложения первых нескольких элементов S
    print("\n[3a] Диагностика: какие элементы S попадают в int(cone{alpha_i})?")
    inside_count = 0
    for v in S:
        if is_strictly_inside_simple_cone_or_opposite(v):
            inside_count += 1
    print(f"    из {len(S)} элементов S во внутренности конуса (или противоположного): "
          f"{inside_count}")
    print(f"    после фильтрации останется: {len(S) - inside_count}")

    # Покажем разложение фундаментальных весов
    print("\n[3b] Разложения omega_i по базису простых корней:")
    for i, w in enumerate(FUND_WEIGHTS):
        c = coords_in_simple_root_basis(w)
        print(f"    omega_{i+1} = {c}")

    print("\n[4] base")
    t0 = time.time()
    Ub, nb = alg_base(S)
    tb = time.time() - t0
    print(f"    троек: {nb}, |U| = {len(Ub)}   {tb:.1f}с")

    print("\n[5] idea1")
    t0 = time.time()
    U1, sf, p1, n1 = alg_idea1(S, W)
    t1 = time.time() - t0
    print(f"    |S'| = {sf}, троек: {n1}, в \\bar C: {p1}")
    print(f"    |U| = {len(U1)}   {t1:.1f}с")

    print("\n[6] сверка")
    print(f"    |Ub| = {len(Ub)}, |U1| = {len(U1)}, совп.: {Ub == U1}")
    print(f"    симм. разность: {len(Ub ^ U1)}")
    print(f"\n    эталон: 37776")

    if Ub != U1:
        diff_only_in_b = Ub - U1
        diff_only_in_1 = U1 - Ub
        print(f"\n    в base, нет в idea1: {len(diff_only_in_b)}")
        print(f"    в idea1, нет в base: {len(diff_only_in_1)}")
        if diff_only_in_b:
            v = next(iter(diff_only_in_b))
            print(f"    пример (только в base): {v}")

    print(f"\n[7] скорость")
    print(f"    base   {tb:.1f}с")
    print(f"    idea1  {t1:.1f}с   ускорение {tb/max(t1, 0.01):.2f}x")


if __name__ == "__main__":
    main()

# хар. набор для D_l, сравнение полного перебора и фильтрации+разнесения

In [ ]:
"""
Построение характеристического набора для D_l двумя способами и
поэлементное сравнение результатов.

  base  — базовый перебор всех C(|S|, l-1) подмножеств в S.
          Точная арифметика sympy. Работает разумно только для D_4.
  idea1 — фильтрация S по знакам разложения по простым корням,
          отбор ядер в \bar C_0, разнесение группой Вейля.
          Версия fast_idea1 использует numpy/float для отсева
          кандидатов и sympy только для финальной точной проверки.

Параметр L внизу. Рекомендуется:
  - L = 4: запустить и base, и idea1, сверить поэлементно.
            Базовый идёт минуты, идея 1 быстрее.
  - L = 5: запустить ТОЛЬКО fast_idea1 (флаг RUN_BASE = False).
            Базовый перебор для D_5 непрактичен (C(162,4) ≈ 7.7e6
            и каждая четвёрка требует точного ядра sympy).
            Эталон для сверки: |U(D_5)| = 46402.
"""

import sympy as sp
import numpy as np
import itertools
import time
from fractions import Fraction
from math import gcd, comb
from functools import reduce


# ─────────────────────────────────────────────────────────────
# Параметры
# ─────────────────────────────────────────────────────────────
L = 4
RUN_BASE = False  # для L=5 поставить False, базовый перебор займёт сутки


# ─────────────────────────────────────────────────────────────
# Простые корни D_l (стандартный выбор):
#   α_i = e_i - e_{i+1}  для i = 1..l-1
#   α_l = e_{l-1} + e_l
# ─────────────────────────────────────────────────────────────
def simple_roots_Dl(l):
    R = []
    for i in range(l-1):
        v = [Fraction(0)] * l
        v[i]   =  Fraction(1)
        v[i+1] = -Fraction(1)
        R.append(v)
    v = [Fraction(0)] * l
    v[l-2] = Fraction(1)
    v[l-1] = Fraction(1)
    R.append(v)
    return R


# ─────────────────────────────────────────────────────────────
# Фундаментальные веса D_l:
#   ω_i = e_1 + ... + e_i           для i = 1..l-2
#   ω_{l-1} = (e_1 + ... + e_{l-1} - e_l) / 2
#   ω_l     = (e_1 + ... + e_{l-1} + e_l) / 2
# ─────────────────────────────────────────────────────────────
def fund_weights_Dl(l):
    out = []
    cur = [Fraction(0)] * l
    for i in range(l-2):
        cur[i] = Fraction(1)
        out.append(list(cur))
    base = list(cur); base[l-2] = Fraction(1)
    w_minus = [x for x in base]; w_minus[l-1] = -Fraction(1)
    w_plus  = [x for x in base]; w_plus[l-1]  =  Fraction(1)
    w_minus = [c/Fraction(2) for c in w_minus]
    w_plus  = [c/Fraction(2) for c in w_plus]
    out.append(w_minus)
    out.append(w_plus)
    return out


# ─────────────────────────────────────────────────────────────
# Все корни D_l (для построения W): ±e_i ± e_j
# ─────────────────────────────────────────────────────────────
def all_roots_Dl(l):
    R = set()
    for i in range(l):
        for j in range(i+1, l):
            for s1 in (1, -1):
                for s2 in (1, -1):
                    v = [Fraction(0)] * l
                    v[i] = Fraction(s1)
                    v[j] = Fraction(s2)
                    R.add(tuple(v))
    return [list(r) for r in R]


# ─────────────────────────────────────────────────────────────
# Нормализация: НОД, знак сохраняется (v и -v — разные элементы)
# ─────────────────────────────────────────────────────────────
def vec_gcd(ints):
    nz = [abs(int(x)) for x in ints if x != 0]
    return reduce(gcd, nz) if nz else 1

def normalize(v):
    fracs = [Fraction(x) for x in v]
    denom = reduce(lambda a, b: a*b//gcd(a, b),
                   [x.denominator for x in fracs])
    ints = [int(x*denom) for x in fracs]
    g = vec_gcd(ints)
    if g == 0:
        return tuple(Fraction(0) for _ in v)
    return tuple(Fraction(x, g) for x in ints)


# ─────────────────────────────────────────────────────────────
# Группа Вейля (BFS от единичной матрицы)
# ─────────────────────────────────────────────────────────────
def reflection_mat(alpha, n):
    nrm = sum(x*x for x in alpha)
    return tuple(tuple(
        (Fraction(1) if i == j else Fraction(0))
        - Fraction(2) * alpha[i] * alpha[j] / nrm
        for j in range(n)) for i in range(n))

def matmul(A, B, n):
    return tuple(tuple(sum(A[i][k]*B[k][j] for k in range(n)) for j in range(n))
                 for i in range(n))

def matvec(M, v, n):
    return [sum(M[i][j]*v[j] for j in range(n)) for i in range(n)]

def build_W(roots, n):
    I = tuple(tuple(Fraction(1) if i == j else Fraction(0)
                    for j in range(n)) for i in range(n))
    refls = [reflection_mat(a, n) for a in roots]
    seen = {I}
    front = [I]
    while front:
        nf = []
        for M in front:
            for r in refls:
                M2 = matmul(r, M, n)
                if M2 not in seen:
                    seen.add(M2)
                    nf.append(M2)
        front = nf
    return list(seen)


# ─────────────────────────────────────────────────────────────
# S = W·{ω_1,...,ω_l}
# ─────────────────────────────────────────────────────────────
def build_S(W, fund, n):
    keys = set()
    for w in fund:
        for M in W:
            keys.add(normalize(matvec(M, w, n)))
    return [list(k) for k in keys]


# ─────────────────────────────────────────────────────────────
# Разложение по простым корням и проверка \bar C_0
# ─────────────────────────────────────────────────────────────
def make_alpha_inv(simple, n):
    A = sp.Matrix([[sp.Rational(simple[j][i].numerator,
                                simple[j][i].denominator)
                    for j in range(n)] for i in range(n)])
    return A.inv()

def coords_in_simple(v, A_inv):
    v_sp = sp.Matrix([sp.Rational(x.numerator, x.denominator) for x in v])
    c = A_inv * v_sp
    return [Fraction(x.p, x.q) for x in c]

def is_inside_alpha_cone(v, A_inv):
    c = coords_in_simple(v, A_inv)
    return all(x > 0 for x in c) or all(x < 0 for x in c)

def in_dom_closure(u, simple, n):
    return all(sum(a[i]*u[i] for i in range(n)) >= 0 for a in simple)


# ─────────────────────────────────────────────────────────────
# Сохранение характеристического набора в файлы
# ─────────────────────────────────────────────────────────────
def save_charset(U, l, prefix=None):
    """
    Сохраняет U в два файла:
      - {prefix}.txt        — точные значения (целые числа после normalize,
                              разделитель «  », совместим с твоим
                              d{L}_characteristic_set.txt из питон-кода)
      - {prefix}_float.txt  — числа с плавающей точкой (через пробел),
                              совместим с save_chambers_covers_Dl.cpp
    Если prefix не задан, используется 'd{l}_characteristic_set'.
    """
    if prefix is None:
        prefix = f"d{l}_characteristic_set"

    # Отсортируем для воспроизводимости
    U_sorted = sorted(U)

    # 1) Точные значения (после normalize все координаты — целые Fraction со знаменателем 1)
    path_exact = f"{prefix}.txt"
    with open(path_exact, "w") as f:
        for v in U_sorted:
            # Каждая координата — Fraction; после normalize знаменатель = 1
            f.write("  ".join(str(c) for c in v) + "\n")
    print(f"  Сохранено {len(U_sorted)} векторов в {path_exact}")

    # 2) Плавающие значения для cpp-кода
    path_float = f"{prefix}_float.txt"
    with open(path_float, "w") as f:
        for v in U_sorted:
            f.write(" ".join(f"{float(c):.1f}" for c in v) + "\n")
    print(f"  Сохранено {len(U_sorted)} векторов в {path_float}")

    return path_exact, path_float


# ─────────────────────────────────────────────────────────────
# Базовый алгоритм: перебор всех C(|S|, l-1) подмножеств в S.
# ВНИМАНИЕ: для D_5 будет долго.
# ─────────────────────────────────────────────────────────────
def alg_base(S, l):
    out = set()
    n_combos = 0
    n_nondeg = 0
    total = comb(len(S), l-1)
    t0 = time.time()
    for combo in itertools.combinations(range(len(S)), l-1):
        n_combos += 1
        if n_combos % 100000 == 0:
            print(f"  base: {n_combos}/{total} ({100*n_combos/total:.1f}%), "
                  f"|U| = {len(out)}, t={time.time()-t0:.0f}с", flush=True)
        rows = [S[i] for i in combo]
        M = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in r]
                       for r in rows])
        if M.rank() < l-1:
            continue
        n_nondeg += 1
        for ns in M.nullspace():
            coords = [Fraction(x.p, x.q) for x in ns]
            out.add(normalize(coords))
            out.add(normalize([-c for c in coords]))
    return out, n_combos, n_nondeg


# ─────────────────────────────────────────────────────────────
# Идея 1 (через sympy на каждой четвёрке — медленная)
# ─────────────────────────────────────────────────────────────
def alg_idea1(S, W, simple, n, l):
    A_inv = make_alpha_inv(simple, n)
    S_filt = [v for v in S if not is_inside_alpha_cone(v, A_inv)]

    partial = set()
    n_combos = 0
    total = comb(len(S_filt), l-1)
    t0 = time.time()
    for combo in itertools.combinations(range(len(S_filt)), l-1):
        n_combos += 1
        if n_combos % 100000 == 0:
            print(f"  idea1: {n_combos}/{total} ({100*n_combos/total:.1f}%), "
                  f"partial={len(partial)}, t={time.time()-t0:.0f}с", flush=True)
        rows = [S_filt[i] for i in combo]
        M = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in r]
                       for r in rows])
        if M.rank() < l-1:
            continue
        for ns in M.nullspace():
            u = [Fraction(x.p, x.q) for x in ns]
            for sign in [+1, -1]:
                u_try = [sign*c for c in u]
                if in_dom_closure(u_try, simple, n):
                    partial.add(normalize(u_try))

    # Разнесение
    print(f"  partial = {len(partial)}, разносим группой W ({len(W)} элем.)")
    full = set()
    for k in partial:
        v = list(k)
        for Mw in W:
            full.add(normalize(matvec(Mw, v, n)))
    return full, len(S_filt), len(partial), n_combos


# ─────────────────────────────────────────────────────────────
# Быстрая идея 1: numpy/float для отсева, sympy только для
# точной финальной проверки кандидатов.
# Это основной алгоритм для D_5.
# ─────────────────────────────────────────────────────────────
def fast_idea1(S, W, simple, n, l):
    A_inv = make_alpha_inv(simple, n)
    S_filt = [v for v in S if not is_inside_alpha_cone(v, A_inv)]
    print(f"  |S'| = {len(S_filt)} (после фильтра)")

    simple_f = [np.array([float(x) for x in a], dtype=np.float64) for a in simple]
    S_filt_f = [np.array([float(x) for x in v], dtype=np.float64) for v in S_filt]
    Sf_np = np.array(S_filt_f)

    partial_keys = set()
    total = comb(len(S_filt), l-1)
    print(f"  всего четвёрок: {total}")

    t0 = time.time()
    n_processed = 0
    n_rank_pass = 0
    n_dom_candidate = 0
    n_added = 0
    EPS = 1e-9

    for combo in itertools.combinations(range(len(S_filt)), l-1):
        n_processed += 1
        if n_processed % 10000 == 0:
            elapsed = time.time() - t0
            rate = n_processed / elapsed if elapsed > 0 else 0
            eta = (total - n_processed) / rate if rate > 0 else 0
            print(f"    {n_processed}/{total} ({100*n_processed/total:.1f}%), "
                  f"elapsed {elapsed:.0f}с, ETA {eta:.0f}с, "
                  f"rank_pass={n_rank_pass}, dom_cand={n_dom_candidate}, added={n_added}",
                  flush=True)

        rows_f = Sf_np[list(combo)]
        rk = np.linalg.matrix_rank(rows_f, tol=EPS)
        if rk < l-1:
            continue
        n_rank_pass += 1

        # Ядро через SVD: правые сингулярные векторы, отвечающие нулевым sv
        _, sv, Vt = np.linalg.svd(rows_f, full_matrices=True)
        kernel_f = Vt[-1]
        # Sanity check, что это действительно ядро
        if np.max(np.abs(rows_f @ kernel_f)) > 1e-6:
            continue

        # Проверяем кандидата на принадлежность \bar C_0 в плавающей точке
        for sign in [+1, -1]:
            u_f = sign * kernel_f
            inner = [a_f @ u_f for a_f in simple_f]
            if all(x > -1e-7 for x in inner):
                n_dom_candidate += 1
                # Точная проверка через sympy
                rows_q = [S_filt[i] for i in combo]
                M = sp.Matrix([[sp.Rational(x.numerator, x.denominator) for x in r]
                               for r in rows_q])
                if M.rank() < l-1:
                    break
                null = M.nullspace()
                if not null:
                    break
                for ns in null:
                    u_q = [Fraction(x.p, x.q) for x in ns]
                    for s2 in [+1, -1]:
                        u_try = [s2*c for c in u_q]
                        if in_dom_closure(u_try, simple, n):
                            key = normalize(u_try)
                            if key not in partial_keys:
                                partial_keys.add(key)
                                n_added += 1
                break

    elapsed = time.time() - t0
    print(f"  Цикл закончен за {elapsed:.0f}с")
    print(f"  rank_pass={n_rank_pass}, dom_cand={n_dom_candidate}, partial={len(partial_keys)}")

    # Разнесение W
    print(f"\n  Разнесение partial={len(partial_keys)} × |W|={len(W)} = {len(partial_keys)*len(W):,}")
    t0 = time.time()
    full = set()
    for cnt, k in enumerate(partial_keys):
        v = list(k)
        for Mw in W:
            full.add(normalize(matvec(Mw, v, n)))
        if (cnt+1) % 5 == 0 or cnt+1 == len(partial_keys):
            print(f"    {cnt+1}/{len(partial_keys)}, |full|={len(full)}, "
                  f"t={time.time()-t0:.0f}с", flush=True)

    return full, len(S_filt), len(partial_keys)


# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────
def main():
    n = L
    print(f"=== D{L} ===\n")

    print("[1] Корни, простые корни, фунд. веса")
    R = all_roots_Dl(n)
    simple = simple_roots_Dl(n)
    fund   = fund_weights_Dl(n)
    print(f"    |R| = {len(R)} (ожидается {2*L*(L-1)})")

    print("\n[2] Группа Вейля")
    t0 = time.time()
    W = build_W(R, n)
    expected_W = 2**(L-1) * reduce(lambda a,b:a*b, range(1, L+1))
    print(f"    |W| = {len(W)} (ожидается {expected_W})    {time.time()-t0:.1f}с")

    print("\n[3] Множество S = W·{ω_i}")
    t0 = time.time()
    S = build_S(W, fund, n)
    print(f"    |S| = {len(S)}    {time.time()-t0:.1f}с")

    print("\n[3a] Диагностика фильтра идеи 1")
    A_inv = make_alpha_inv(simple, n)
    inside = sum(1 for v in S if is_inside_alpha_cone(v, A_inv))
    print(f"    в int(±cone{{α}}): {inside}")
    print(f"    после фильтрации: {len(S) - inside}")

    Ub = None
    if RUN_BASE:
        print("\n[4] Базовый перебор")
        t0 = time.time()
        Ub, nb, ndb = alg_base(S, L)
        tb = time.time() - t0
        print(f"    троек: {nb}, невырожденных: {ndb}")
        print(f"    |U_base| = {len(Ub)}    {tb:.1f}с")
    else:
        print("\n[4] Базовый перебор ПРОПУЩЕН (RUN_BASE = False)")

    print("\n[5] Быстрая идея 1 + разнесение")
    t0 = time.time()
    U1, sf, p1 = fast_idea1(S, W, simple, n, L)
    t1 = time.time() - t0
    print(f"    |U_idea1| = {len(U1)}    {t1:.0f}с")

    if Ub is not None:
        print("\n[6] Поэлементное сравнение")
        print(f"    |U_base|  = {len(Ub)}")
        print(f"    |U_idea1| = {len(U1)}")
        print(f"    совпадают как множества: {Ub == U1}")
        print(f"    |U_base \\ U_idea1| = {len(Ub - U1)}")
        print(f"    |U_idea1 \\ U_base| = {len(U1 - Ub)}")
        print(f"    |U_base ∩ U_idea1| = {len(Ub & U1)}")
        if Ub == U1:
            print("\n    Идея 1 + разнесение даёт ровно тот же набор, что и базовый")
        else:
            print("\n    РАСХОЖДЕНИЕ — нужно разбираться")
            if Ub - U1:
                v = next(iter(Ub - U1))
                print(f"      есть в base, нет в idea1: {v}")
            if U1 - Ub:
                v = next(iter(U1 - Ub))
                print(f"      есть в idea1, нет в base: {v}")
    else:
        if L == 5:
            print(f"\n[6] Сверка с эталоном")
            print(f"    |U_idea1| = {len(U1)}")
            print(f"    эталон для D_5: 46402")
            print(f"    совпадает: {'✓' if len(U1) == 46402 else '✗'}")

    # Сохранение в файлы
    print(f"\n[7] Сохранение характеристического набора")
    save_charset(U1, L)


if __name__ == "__main__":
    main()